In [0]:
import importlib, sys
importlib.invalidate_caches()
for name in list(sys.modules):
    if name.startswith("common_utils"):
        del sys.modules[name]

In [0]:
dbutils.library.restartPython()

In [0]:

dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")
     

In [0]:
import json
from datetime import date

from common_utils.bronze import read_raw, bronze_ingestor
from common_utils.logging import get_logger


# ============================================================
# 0. LOGGER
# ============================================================

logger = get_logger("ds2b")


# ============================================================
# 1. CONFIG PATH
# ============================================================

dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")

logger.info("Reading Bronze configuration from %s", config_path)


# ============================================================
# 2. READ CONFIG
# ============================================================

with open(config_path, "r") as f:
    config = json.load(f)


# ============================================================
# 3. RUN DATE
# ============================================================

run_date = date.today().isoformat()

logger.info("Bronze ingestion run date: %s", run_date)


# ============================================================
# 4. BASE RAW PATH
# ============================================================

base_raw_path = "/Volumes/retaildata_platform/bronze/raw_data"

logger.info("Base raw path: %s", base_raw_path)


# ============================================================
# 5. PROCESS EACH SOURCE
# ============================================================

for source_name, source_config in config.items():

    raw_folder = source_config["raw_folder"]
    file_format = source_config["file_format"]
    read_options = source_config.get("read_options", {})
    target_table = source_config["target_table"]
    mode = source_config["mode"]

    # --------------------------------------------------------
    # Build raw path
    # --------------------------------------------------------

    raw_path = (
        f"{base_raw_path}/"
        f"{raw_folder}/"
        f"load_date={run_date}"
    )

    logger.info(
        "[%s] Starting Bronze ingestion",
        source_name
    )

    logger.info(
        "[%s] Reading raw data from %s",
        source_name,
        raw_path
    )

    # --------------------------------------------------------
    # Read Raw
    # --------------------------------------------------------

    df = read_raw(
        spark,
        raw_path,
        file_format=file_format,
        options=read_options
    )

    logger.info(
        "[%s] Raw data successfully read",
        source_name
    )

    # --------------------------------------------------------
    # Bronze ingestion
    # --------------------------------------------------------

    rows = bronze_ingestor(
        df,
        mode=mode,
        target_table=target_table
    )

    logger.info(
        "[%s] Successfully ingested %s rows into %s",
        source_name,
        rows,
        target_table
    )


# ============================================================
# 6. COMPLETION
# ============================================================

logger.info("Bronze ingestion finished successfully for all sources")

2026-10-28 12:23:50,744 INFO Reading Bronze configuration from ../config/config_ds2b.json
2026-10-28 12:23:50,785 INFO Bronze ingestion run date: 2026-10-28
2026-10-28 12:23:50,786 INFO Base raw path: /Volumes/retaildata_platform/bronze/raw_data
2026-10-28 12:23:50,787 INFO [s3_sales] Starting Bronze ingestion
2026-10-28 12:23:50,788 INFO [s3_sales] Reading raw data from /Volumes/retaildata_platform/bronze/raw_data/s3_sales/load_date=2026-10-28
2026-10-28 12:23:50,788 INFO Reading from path : /Volumes/retaildata_platform/bronze/raw_data/s3_sales/load_date=2026-10-28
2026-10-28 12:23:50,788 INFO File Format : parquet
2026-10-28 12:23:50,789 INFO [s3_sales] Raw data successfully read
2026-10-28 12:23:50,789 INFO Adding Columns in dataframe
2026-10-28 12:23:50,790 INFO Writing Delta Table : retaildata_platform.bronze.s3_sales
2026-10-28 12:23:50,791 INFO Selected mode is  : overwrite
2026-10-28 12:23:54,321 INFO [s3_sales] Successfully ingested 360 rows into retaildata_platform.bronze.s3_

In [0]:
display(spark.table("retaildata_platform.bronze.sqlserver_customers").limit(5))

customer_id,tax_id,tax_code,customer_name,state,city,postcode,street,number,unit,region,district,lon,lat,ship_to_address,valid_from,valid_to,units_purchased,loyalty_segment,last_update_ts,file_path
11123757,NULL,NULL,"SMITH, SHIRLEY",IN,BREMEN,46506.0,N CENTER ST,521.0,NULL,Indiana,50.0,-86.1465825,41.4507625,"IN, 46506.0, N CENTER ST, 521.0",1532824233,1548137353,34,3,2026-10-28T12:24:00.158Z,dbfs:/Volumes/retaildata_platform/bronze/raw_data/sqlserver_customers/load_date=2026-10-28/part-00000-tid-6159667814481937359-bbd48f80-d119-4d47-ab19-e3076cb4234c-150-1-c000.csv
10061536,268043981,A,ellevation education,NJ,PARAMUS,7652.0,Linden Street,631,NULL,NULL,NULL,-74.0810309,40.9659824,"NJ, 7652.0, Linden Street, 631",1533291565,NULL,8,2,2026-10-28T12:24:00.158Z,dbfs:/Volumes/retaildata_platform/bronze/raw_data/sqlserver_customers/load_date=2026-10-28/part-00000-tid-6159667814481937359-bbd48f80-d119-4d47-ab19-e3076cb4234c-150-1-c000.csv
14567930,NULL,NULL,"BERES, CHRIS L",CA,Long Beach,90804.0,State Route 1,4775,6,NULL,NULL,-118.1371939,33.7867492,"CA, 90804.0, State Route 1, 4775",1529783767,NULL,2,0,2026-10-28T12:24:00.158Z,dbfs:/Volumes/retaildata_platform/bronze/raw_data/sqlserver_customers/load_date=2026-10-28/part-00000-tid-6159667814481937359-bbd48f80-d119-4d47-ab19-e3076cb4234c-150-1-c000.csv
1005505,NULL,NULL,"HARRIS, MICHAEL K",MA,YARMOUTH,2673.0,ROUTE 28,164,234,MA,BARNSTABLE,-70.25878780000001,41.6580657,"MA, 2673.0, ROUTE 28, 164",1543015928,NULL,3,0,2026-10-28T12:24:00.158Z,dbfs:/Volumes/retaildata_platform/bronze/raw_data/sqlserver_customers/load_date=2026-10-28/part-00000-tid-6159667814481937359-bbd48f80-d119-4d47-ab19-e3076cb4234c-150-1-c000.csv
30585978,NULL,NULL,"STEPHENS, GERALDINE M",OR,ADDRESS,0,NO SITUS,NULL,NULL,NULL,NULL,-122.1055158,45.374317,"OR, 0, NO SITUS, nan",1523100473,NULL,18,3,2026-10-28T12:24:00.158Z,dbfs:/Volumes/retaildata_platform/bronze/raw_data/sqlserver_customers/load_date=2026-10-28/part-00000-tid-6159667814481937359-bbd48f80-d119-4d47-ab19-e3076cb4234c-150-1-c000.csv


In [0]:
display(dbutils.fs.ls("/Volumes/retaildata_platform/bronze/raw_data/sqlserver_customers/load_date=2026-10-28"))

path,name,size,modificationTime
dbfs:/Volumes/retaildata_platform/bronze/raw_data/sqlserver_customers/load_date=2026-10-28/_SUCCESS,_SUCCESS,0,1793188934000
dbfs:/Volumes/retaildata_platform/bronze/raw_data/sqlserver_customers/load_date=2026-10-28/_committed_6159667814481937359,_committed_6159667814481937359,212,1793188933000
dbfs:/Volumes/retaildata_platform/bronze/raw_data/sqlserver_customers/load_date=2026-10-28/_committed_6828387682597455784,_committed_6828387682597455784,113,1793188906000
dbfs:/Volumes/retaildata_platform/bronze/raw_data/sqlserver_customers/load_date=2026-10-28/_started_6159667814481937359,_started_6159667814481937359,0,1793188930000
dbfs:/Volumes/retaildata_platform/bronze/raw_data/sqlserver_customers/load_date=2026-10-28/_started_6828387682597455784,_started_6828387682597455784,0,1793188902000
dbfs:/Volumes/retaildata_platform/bronze/raw_data/sqlserver_customers/load_date=2026-10-28/part-00000-tid-6159667814481937359-bbd48f80-d119-4d47-ab19-e3076cb4234c-150-1-c000.csv,part-00000-tid-6159667814481937359-bbd48f80-d119-4d47-ab19-e3076cb4234c-150-1-c000.csv,5979875,1793188933000
